# Day 09. Exercise 02
# Metrics

## 0. Imports

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import joblib
import warnings
warnings.filterwarnings('ignore')

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [14]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
dow = pd.read_csv('../data/dayofweek.csv')
df['dayofweek'] = dow['dayofweek']
print(df.shape)
df.head()

(1686, 44)


,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1,dayofweek
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4


In [15]:
X = df.drop(columns=['dayofweek'])
y = df['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

X_train: (1348, 43), X_test: (338, 43)


## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [16]:
def compute_metrics(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_proba, multi_class='ovo', average='weighted')

    print(f"accuracy is {acc:.5f}")
    print(f"precision is {prec:.5f}")
    print(f"recall is {rec:.5f}")
    print(f"roc_auc is {auc:.5f}")

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'roc_auc': auc, 'model': model, 'y_pred': y_pred}

In [17]:
svm = SVC(C=10, class_weight=None, gamma='auto', kernel='rbf',
          probability=True, random_state=21)
svm_metrics = compute_metrics(svm, X_train, y_train, X_test, y_test)

accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878


## 3. Decision tree

1. The same task for decision tree

In [18]:
tree = DecisionTreeClassifier(class_weight='balanced', criterion='gini',
                              max_depth=22, random_state=21)
tree_metrics = compute_metrics(tree, X_train, y_train, X_test, y_test)

accuracy is 0.89053
precision is 0.89262
recall is 0.89053
roc_auc is 0.93664


## 4. Random forest

1. The same task for random forest.

In [19]:
rf = RandomForestClassifier(class_weight=None, criterion='gini',
                            max_depth=28, n_estimators=50, random_state=21)
rf_metrics = compute_metrics(rf, X_train, y_train, X_test, y_test)

accuracy is 0.92899
precision is 0.93009
recall is 0.92899
roc_auc is 0.99033


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [20]:
print('Сравнение моделей')
all_metrics = {'SVM': svm_metrics, 'Decision Tree': tree_metrics, 'Random Forest': rf_metrics}
for name, m in all_metrics.items():
    print(f"{name:15s} - acc={m['accuracy']:.5f}, prec={m['precision']:.5f}, rec={m['recall']:.5f}, auc={m['roc_auc']:.5f}")

best_name = max(all_metrics, key=lambda k: np.mean([all_metrics[k]['accuracy'], all_metrics[k]['precision'],
                                                    all_metrics[k]['recall'], all_metrics[k]['roc_auc']
                                                    ]))
print(f"\nЛучшая модель: {best_name}")
best_model = all_metrics[best_name]['model']
best_y_pred = all_metrics[best_name]['y_pred']

Сравнение моделей
SVM             - acc=0.88757, prec=0.89267, rec=0.88757, auc=0.97878
Decision Tree   - acc=0.89053, prec=0.89262, rec=0.89053, auc=0.93664
Random Forest   - acc=0.92899, prec=0.93009, rec=0.92899, auc=0.99033

Лучшая модель: Random Forest


In [21]:
days = {0: 'Понедельник', 1: 'Вторник', 2: 'Среда', 3: 'Четверг',
       4: 'Пятница', 5: 'Суббота', 6: 'Воскресенье'}

df_full = df.copy()
df_full['labname'] = df_full.filter(like='labname_').idxmax(axis=1).str.replace('labname_', '')
df_full['uid'] = df_full.filter(like='uid_').idxmax(axis=1).str.replace('uid_', '')

df_test = df_full.loc[X_test.index].copy()
df_test['y_pred'] = best_y_pred
df_test['error'] = (df_test['dayofweek'] != df_test['y_pred']).astype(int)

totals = df_full.groupby('dayofweek').size()
errors = df_test.groupby('dayofweek')['error'].sum()
day_pct = (errors / totals * 100).sort_index()

print("Ошибки по дням недели (% от числа образцов класса в полном датасете)")
for d, pct in day_pct.items():
    print(f"  {days[d]:15s}- {errors[d]:3.0f} ошибок из {totals[d]:3d} значений = {pct:.1f}%")
print(f"\nБольше всего ошибок: {days[day_pct.idxmax()]} ({day_pct.max():.1f}%)")

lab_pct = (df_test.groupby('labname')['error'].sum() / df_full.groupby('labname').size() * 100).sort_values(ascending=False)
print("\nОшибки по labname (топ-5)")
print(lab_pct.rename_axis(None).head(5).to_string())
print(f"\nБольше всего ошибок: {lab_pct.idxmax()} ({lab_pct.max():.1f}%)")

user_pct = (df_test.groupby('uid')['error'].sum() / df_full.groupby('uid').size() * 100).sort_values(ascending=False)
print("\nОшибки по пользователям (топ-5)")
print(user_pct.rename_axis(None).head(5).to_string())
print(f"\nБольше всего ошибок: {user_pct.idxmax()} ({user_pct.max():.1f}%)")


Ошибки по дням недели (% от числа образцов класса в полном датасете)
  Понедельник    -   7 ошибок из 136 значений = 5.1%
  Вторник        -   6 ошибок из 274 значений = 2.2%
  Среда          -   2 ошибок из 149 значений = 1.3%
  Четверг        -   2 ошибок из 396 значений = 0.5%
  Пятница        -   3 ошибок из 104 значений = 2.9%
  Суббота        -   3 ошибок из 271 значений = 1.1%
  Воскресенье    -   1 ошибок из 356 значений = 0.3%

Больше всего ошибок: Понедельник (5.1%)

Ошибки по labname (топ-5)
lab03      100.000000
lab03s     100.000000
laba06       4.166667
laba04       3.370787
laba06s      3.278689

Больше всего ошибок: lab03 (100.0%)

Ошибки по пользователям (топ-5)
user_22    14.285714
user_6      8.333333
user_16     6.250000
user_19     4.395604
user_27     4.347826

Больше всего ошибок: user_22 (14.3%)


In [22]:
joblib.dump(best_model, 'best_by_metrics.sav')

['best_by_metrics.sav']

## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [23]:
def evaluate_models(models, params_list, X_train, y_train, X_test, y_test):
    results = {}
    for model_cls, params in zip(models, params_list):
        model = model_cls(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted'),
            'recall': recall_score(y_test, y_pred, average='weighted'),
            'roc_auc': roc_auc_score(y_test, y_proba, multi_class='ovo', average='weighted')
        }
        results[model_cls.__name__] = metrics
        
    return results


models = [SVC, DecisionTreeClassifier, RandomForestClassifier]
params_list = [
    {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf',
     'probability': True, 'random_state': 21},
    {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 22,
     'random_state': 21},
    {'class_weight': None, 'criterion': 'gini', 'max_depth': 28,
     'n_estimators': 50, 'random_state': 21}
]

all_results = evaluate_models(models, params_list, X_train, y_train, X_test, y_test)
print(all_results)

{'SVC': {'accuracy': 0.8875739644970414, 'precision': 0.8926729169690374, 'recall': 0.8875739644970414, 'roc_auc': np.float64(0.9787793228216216)}, 'DecisionTreeClassifier': {'accuracy': 0.8905325443786982, 'precision': 0.8926192681313897, 'recall': 0.8905325443786982, 'roc_auc': np.float64(0.9366351447213223)}, 'RandomForestClassifier': {'accuracy': 0.9289940828402367, 'precision': 0.9300865038851309, 'recall': 0.9289940828402367, 'roc_auc': np.float64(0.9903274757720744)}}
